# Task 11: Ex-post Attribution

In [1]:
import numpy as np
import pandas as pd

In [5]:
# --- 11.1: Ex-post Attribution (asset level) ---
returns   = pd.read_csv('test11_1_returns.csv')
weights_s = pd.read_csv('test11_1_weights.csv')
w = weights_s['W'].values
R = returns.values
T, n_assets = R.shape
assets = returns.columns.tolist()
print("Assets:", assets, "  Weights:", w, "  T:", T)

# ---- Buy-and-hold portfolio ----
cum_val = np.zeros((T+1, n_assets))
cum_val[0] = w                                  # initial $ allocation
for t in range(T):
    cum_val[t+1] = cum_val[t] * (1 + R[t])
port_val = cum_val.sum(axis=1)                  # portfolio value path
w_dyn    = cum_val[:-1] / port_val[:-1, None]   # time-varying weights (T x n)
r_p_bah  = (w_dyn * R).sum(axis=1)             # BaH period returns
contrib  = w_dyn * R                            # per-asset BaH contributions (T x n)

# ---- Total Returns (geometric) ----
geo      = (1 + R).prod(axis=0) - 1            # per asset
port_geo = port_val[-1] - 1                     # BaH portfolio geo (= w @ geo)

# ---- Return Attribution ----
ret_attr = w * geo                              # w_i * TotalReturn_i

# ---- Vol Attribution ----
port_std = np.std(r_p_bah, ddof=1)
vol_attr = np.array([
    np.cov(contrib[:, i], r_p_bah, ddof=1)[0, 1] / port_std
    for i in range(n_assets)
])
port_vol = vol_attr.sum()

result_11_1 = pd.DataFrame({
    'Value': ['TotalReturn', 'Return Attribution', 'Vol Attribution'],
    **{a: [geo[i], ret_attr[i], vol_attr[i]] for i, a in enumerate(assets)},
    'Portfolio': [port_geo, port_geo, port_vol],
})
print("\n", result_11_1.to_string(index=False))
result_11_1.to_csv('testout_11.1.csv', index=False)
print("\nSaved testout_11.1.csv")

Assets: ['x1', 'x2', 'x3']   Weights: [0.3 0.2 0.5]   T: 30

              Value        x1        x2       x3  Portfolio
       TotalReturn -0.221446 -0.016008 0.301467   0.081098
Return Attribution -0.066434 -0.003202 0.150733   0.081098
   Vol Attribution -0.000620  0.002827 0.012580   0.014787

Saved testout_11.1.csv


In [7]:
# --- 11.2: Ex-post Attribution to Factors (BaH portfolio) ---
factor_ret  = pd.read_csv('test11_2_factor_returns.csv')
stock_ret   = pd.read_csv('test11_2_stock_returns.csv')
beta_df     = pd.read_csv('test11_2_beta.csv', index_col=0)
weights_df  = pd.read_csv('test11_2_weights.csv')

w = weights_df['W'].values
F = factor_ret.values
R = stock_ret.values
B = beta_df.values
T, n_stocks  = R.shape
n_factors    = F.shape[1]
factors      = factor_ret.columns.tolist()
print("Stocks:", beta_df.index.tolist(), "  Weights:", w)
print("Factors:", factors)

# ---- Buy-and-hold portfolio ----
cum_val = np.zeros((T+1, n_stocks))
cum_val[0] = w
for t in range(T):
    cum_val[t+1] = cum_val[t] * (1 + R[t])
port_val = cum_val.sum(axis=1)
w_dyn    = cum_val[:-1] / port_val[:-1, None]
r_p_bah  = (w_dyn * R).sum(axis=1)

# ---- Per-period factor and alpha contributions (BaH) ----
alpha_stock = R - F @ B.T
E_j_t       = w_dyn @ B
factor_c    = E_j_t * F
alpha_c     = (w_dyn * alpha_stock).sum(axis=1)

# Verify decomposition
assert np.allclose(r_p_bah, factor_c.sum(axis=1) + alpha_c, atol=1e-10), "Decomposition mismatch"
print("Decomposition check passed.")

# ---- Total Returns (geometric) ----
geo_factors  = (1 + F).prod(axis=0) - 1
geo_alpha    = np.prod(1 + alpha_c) - 1
port_geo     = port_val[-1] - 1

# ---- Return Attribution (arithmetic sum of BaH contributions) ----
factor_ret_attr = factor_c.sum(axis=0)            # sum of period contributions
alpha_ret_attr  = alpha_c.sum()
port_ret_attr   = r_p_bah.sum()                   # arithmetic portfolio total

# ---- Vol Attribution ----
port_std = np.std(r_p_bah, ddof=1)
factor_vol_attr = np.array([
    np.cov(factor_c[:, j], r_p_bah, ddof=1)[0, 1] / port_std
    for j in range(n_factors)
])
alpha_vol_attr = np.cov(alpha_c, r_p_bah, ddof=1)[0, 1] / port_std
port_vol = factor_vol_attr.sum() + alpha_vol_attr

result_11_2 = pd.DataFrame({
    'Value': ['TotalReturn', 'Return Attribution', 'Vol Attribution'],
    **{factors[j]: [geo_factors[j], factor_ret_attr[j], factor_vol_attr[j]]
       for j in range(n_factors)},
    'Alpha':    [geo_alpha,       alpha_ret_attr,   alpha_vol_attr],
    'Portfolio':[port_geo,        port_ret_attr,    port_vol],
})
print("\n", result_11_2.to_string(index=False))
result_11_2.to_csv('testout_11.2.csv', index=False)
print("\nSaved testout_11.2.csv")

Stocks: ['S1', 'S2']   Weights: [0.5 0.5]
Factors: ['F1', 'F2', 'F3']
Decomposition check passed.

              Value       F1        F2        F3    Alpha  Portfolio
       TotalReturn 0.079939  0.026711 -0.016317 0.046443   0.105327
Return Attribution 0.065020  0.000159 -0.006131 0.048529   0.107577
   Vol Attribution 0.005013 -0.000009  0.001785 0.005445   0.012234

Saved testout_11.2.csv
